In [1]:
!pip install -q sentence-transformers faiss-cpu scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 95.8 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving listings_prepared.csv to listings_prepared.csv
Saving retrieval.py to retrieval.py


In [3]:
from retrieval import Schema, FieldSpec, Corpus, SemanticRetriever, SentenceTransformerEmbedder

schema = Schema(
    id_field="id",
    text_spec=[FieldSpec("embed_text")],
    metadata_fields=["price_num", "room_type", "neighbourhood_cleansed"],
)

engine = SemanticRetriever(schema, SentenceTransformerEmbedder())
engine.build(Corpus.from_csv("listings_prepared.csv", schema))
print("docs:", len(engine.corpus))

hits = engine.search("quiet family home near the park", k=5,
                     filters={"price_num": {"lte": 200}})
for h in hits:
    print(round(h["score"], 3), h["text"].split(chr(10))[0], h["metadata"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

docs: 7528
0.717 Title: Quiet Garden Apartment {'price_num': 179.43, 'room_type': 'Entire home/apt', 'neighbourhood_cleansed': 'Parkside'}
0.683 Title: Perfect for someone loving the park + ocean vibe. {'price_num': 116.97, 'room_type': 'Private room', 'neighbourhood_cleansed': 'Outer Sunset'}
0.679 Title: Quiet, Relaxing House Close to BART Station {'price_num': 185.51, 'room_type': 'Entire home/apt', 'neighbourhood_cleansed': 'Ocean View'}
0.679 Title: Garden cottage on golden gate park {'price_num': 150.9, 'room_type': 'Entire home/apt', 'neighbourhood_cleansed': 'Inner Richmond'}
0.677 Title: LuxoStays | ! Quiet Rm #Private Bathrm & VIEW {'price_num': 73.3, 'room_type': 'Private room', 'neighbourhood_cleansed': 'Excelsior'}


In [4]:
from google.colab import files
files.upload()   # evaluate.py

Saving evaluate.py to evaluate.py


{'evaluate.py': b'"""\nevaluate.py \xe2\x80\x94 \xe9\x80\x9a\xe7\x94\xa8\xe6\xa3\x80\xe7\xb4\xa2\xe8\xaf\x84\xe4\xbc\xb0\xe5\x99\xa8:Recall@K / NDCG@K / MRR\n\n\xe4\xb8\x8d\xe4\xbe\x9d\xe8\xb5\x96\xe4\xbb\xbb\xe4\xbd\x95\xe6\xa8\xa1\xe5\x9e\x8b,\xe5\x8f\xaa\xe8\xa6\x81\xe6\xa3\x80\xe7\xb4\xa2\xe5\x99\xa8\xe5\xae\x9e\xe7\x8e\xb0 .search(query, k) -> [{"id": ...}, ...] \xe5\x8d\xb3\xe5\x8f\xaf\xe3\x80\x82\n\xe6\x89\x80\xe4\xbb\xa5\xe5\xaf\xb9\xe4\xbd\xa0\xe7\x9a\x84 TF-IDF \xe5\xbc\x95\xe6\x93\x8e\xe3\x80\x81\xe8\xaf\xad\xe4\xb9\x89\xe5\xbc\x95\xe6\x93\x8e\xe3\x80\x81\xe4\xbb\xa5\xe5\x90\x8e\xe7\x9a\x84 Hybrid \xe5\xbc\x95\xe6\x93\x8e\xe9\x83\xbd\xe9\x80\x9a\xe7\x94\xa8\xe3\x80\x82\n\nqrels(\xe8\xaf\x84\xe6\xb5\x8b\xe9\x9b\x86)\xe6\xa0\xbc\xe5\xbc\x8f:\n    {\xe6\x9f\xa5\xe8\xaf\xa2\xe5\xad\x97\xe7\xac\xa6\xe4\xb8\xb2: {\xe7\x9b\xb8\xe5\x85\xb3\xe6\x88\xbf\xe6\xba\x90id, ...}, ...}\n"""\n\nimport math\n\n\ndef _dcg(rels):\n    """\xe6\x8a\x98\xe6\x8d\x9f\xe7\xb4\xaf\xe8\xae\xa1\xe5\xa2\x

In [5]:
from retrieval import TfidfEmbedder
engine_lex = SemanticRetriever(schema, TfidfEmbedder(max_features=4000))
engine_lex.build(Corpus.from_csv("listings_prepared.csv", schema))
print("semantic base done")

semantic base done


In [6]:
import pandas as pd, re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

df = pd.read_csv("listings_prepared.csv").dropna(subset=["embed_text"])
sample = df.sample(80, random_state=0)

tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to("cuda")


def listing_for_query(text):
    lines = [l for l in text.split("\n") if l.startswith(("Title:", "Description:"))]
    return " ".join(lines)[:400]

def make_query(text):
    prompt = ("You are a traveler searching for a place to stay. Based on the listing below, "
              "write ONE short search query (6-10 words) describing what you want, in your own words. "
              "Do NOT copy the listing's title or sentences. Output only the query.\n\nListing: "
              + text + "\n\nSearch query:")
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids.to("cuda")
    out = model.generate(ids, max_new_tokens=24, do_sample=True, temperature=0.7)
    return tok.decode(out[0], skip_special_tokens=True).strip()

def looks_clean(q):
    if len(q.split()) < 4: return False
    if len(re.findall(r"\d", q)) > 4: return False
    if "/" in q or "sqft" in q.lower(): return False
    return True

qrels = {}
for _, row in sample.iterrows():
    q = make_query(listing_for_query(row["embed_text"]))
    if q and looks_clean(q) and q not in qrels:
        qrels[q] = {row["id"]}

print(f"Generated {len(qrels)} queries. Samples:")
for q in list(qrels)[:8]:
    print("  -", q)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generated 79 queries. Samples:
  - i want to see a room with a private bath in sf
  - find the city, state, or country with the most business in the city and the most businesses in the county.
  - Find a warm, cozy room in the city close to San Francisco State, SF Town
  - Find a place with a view of the bay.
  - If you are looking for a 1 bedroom, or 2 bedroom in the financial district, in Union Square, near
  - Find "two full room - 2 beds"
  - The city i want to stay in is San Francisco, please show me a studio that has the following amenities:
  - Find a room to rent in Mission Central, San Francisco, CA. If it's near the Valencia and Hay


In [7]:
import re
bad_starts = ("what", "list", "how", "where", "when", "is ", "are ", "does")
def stricter(q):
    ql = q.lower().strip()
    if ql.startswith(bad_starts): return False
    if '"' in q: return False
    if re.search(r"[A-Z]\.[A-Z]", q): return False
    return True

qrels = {q: ids for q, ids in qrels.items() if stricter(q)}
print(f"after filtering {len(qrels)} remains:")
for q in list(qrels)[:8]:
    print("  -", q)

after filtering 66 remains:
  - i want to see a room with a private bath in sf
  - find the city, state, or country with the most business in the city and the most businesses in the county.
  - Find a warm, cozy room in the city close to San Francisco State, SF Town
  - Find a place with a view of the bay.
  - If you are looking for a 1 bedroom, or 2 bedroom in the financial district, in Union Square, near
  - The city i want to stay in is San Francisco, please show me a studio that has the following amenities:
  - Find a room to rent in Mission Central, San Francisco, CA. If it's near the Valencia and Hay
  - Find a place to stay in SF, CA with a view of the Golden Gate Bridge.


In [8]:
"""
generate_qrels.py

Generate synthetic search queries and build qrels for offline
retrieval evaluation.

Pipeline:
    listing -> FLAN-T5 query generation -> quality filtering -> qrels

The resulting benchmark can be used to evaluate:
    - Recall@K
    - Precision@K
    - NDCG
    - MRR
"""

import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load retrieval corpus and sample listings for evaluation.
df = pd.read_csv("listings_prepared.csv").dropna(subset=["embed_text"])
sample = df.sample(300, random_state=0)

# Instruction-tuned model for query generation.
tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-large"
).to("cuda")


def listing_for_query(text):
    """
    Keep only title and description to focus generation
    on user intent rather than structured metadata.
    """
    lines = [
        line
        for line in text.split("\n")
        if line.startswith(("Title:", "Description:"))
    ]
    return " ".join(lines)[:400]


def make_query(text):
    """
    Generate a short traveler-style search query.
    """
    prompt = (
        "You are a traveler searching for a place to stay. "
        "Based on the listing below, write ONE short search "
        "query (6-10 words) describing what you want, in your "
        "own words. Do NOT copy the listing's title or sentences. "
        "Output only the query.\n\n"
        f"Listing: {text}\n\n"
        "Search query:"
    )

    ids = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids.to("cuda")

    outputs = model.generate(
        ids,
        max_new_tokens=24,
        do_sample=True,
        temperature=0.7
    )

    return tok.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()


# Heuristic filters for removing low-quality generations.
bad_starts = (
    "what", "list", "how", "where",
    "when", "is ", "are ", "does"
)


def is_clean(query):
    """
    Retain only realistic search-style queries.
    """
    q = query.lower().strip()

    if len(query.split()) < 4:
        return False

    if len(re.findall(r"\d", query)) > 4:
        return False

    if "/" in query or "sqft" in q:
        return False

    if q.startswith(bad_starts):
        return False

    if '"' in query:
        return False

    if re.search(r"[A-Z]\.[A-Z]", query):
        return False

    return True


# Build query relevance judgments (qrels).
qrels = {}

for n, (_, row) in enumerate(sample.iterrows(), start=1):

    query = make_query(
        listing_for_query(row["embed_text"])
    )

    if query and is_clean(query) and query not in qrels:
        qrels[query] = {row["id"]}

    if n % 50 == 0:
        print(
            f"Processed {n}/300 "
            f"| Retained {len(qrels)} queries"
        )

print(f"\nGenerated {len(qrels)} clean queries.\n")

for query in list(qrels)[:8]:
    print(" -", query)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Processed 50/300 | Retained 37 queries
Processed 100/300 | Retained 77 queries
Processed 150/300 | Retained 118 queries
Processed 200/300 | Retained 159 queries
Processed 250/300 | Retained 198 queries
Processed 300/300 | Retained 240 queries

Generated 240 clean queries.

 - Find a place to stay near Golden Gate Park
 - Find a house in San Jose, California near the Ferry Building.
 - Located in the financial district, this studio is near shops, restaurants, and Moscone Center.
 - view all the places in san francisco
 - Find a hotel in Mission, San Francisco.
 - Find a room in downtown SF with a view of the city & the bay.
 - Find a place to stay in San Francisco.
 - Find a 3 bedroom hotel room at the Marriott Vacation Club in San Francisco and book it through booking.com


In [9]:
from evaluate import compare
compare({"lexical (TF-IDF)": engine_lex, "semantic (bge)": engine}, qrels, k=10)


retriever          Recall@10     NDCG@10         MRR
----------------------------------------------------
lexical (TF-IDF)       0.350       0.227       0.189
semantic (bge)         0.450       0.340       0.306


{'lexical (TF-IDF)': {'Recall@10': 0.35,
  'NDCG@10': 0.2266918544970872,
  'MRR': 0.18914021164021164},
 'semantic (bge)': {'Recall@10': 0.45,
  'NDCG@10': 0.34027296625920733,
  'MRR': 0.3056812169312169}}

In [10]:
from collections import defaultdict


class HybridRetriever:
    """
    Hybrid retrieval using Reciprocal Rank Fusion (RRF).

    RRF combines rankings from multiple retrievers without requiring
    score normalization. It is commonly used to fuse lexical and
    semantic search results.

    Formula:
        score(d) = Σ 1 / (rrf_k + rank_d)

    Typical setup:
        TF-IDF / BM25 + Dense Retrieval
    """

    def __init__(self, retrievers, rrf_k=60, pool=50):
        """
        Args:
            retrievers:
                List of retrieval engines to fuse.

            rrf_k:
                Rank smoothing parameter. Larger values reduce
                the impact of top-ranked documents.

            pool:
                Number of candidates retrieved from each engine
                before fusion.
        """
        self.retrievers = retrievers
        self.rrf_k = rrf_k
        self.pool = pool

    def search(self, query, k=10):
        """
        Execute hybrid retrieval and return top-k fused results.
        """

        scores = defaultdict(float)
        meta = {}

        for retriever in self.retrievers:

            # Collect candidates from each retrieval model.
            results = retriever.search(
                query,
                k=self.pool
            )

            for rank, hit in enumerate(results):

                # Reciprocal Rank Fusion score.
                scores[hit["id"]] += (
                    1.0 / (self.rrf_k + rank + 1)
                )

                meta[hit["id"]] = hit

        ranked_ids = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                "id": doc_id,
                "score": scores[doc_id],
                **{
                    field: meta[doc_id][field]
                    for field in ("text", "metadata")
                    if field in meta[doc_id]
                },
            }
            for doc_id in ranked_ids
        ]


# -------------------------------------------------------
# Hybrid retrieval:
#     Dense Retrieval (BGE)
#   + Lexical Retrieval (TF-IDF)
#   + Reciprocal Rank Fusion
# -------------------------------------------------------
hybrid = HybridRetriever(
    [engine, engine_lex]
)

# -------------------------------------------------------
# Offline evaluation using synthetic qrels.
#
# Metrics:
#     Recall@K
#     Precision@K
#     NDCG@K
# -------------------------------------------------------
from evaluate import compare

compare(
    {
        "lexical (TF-IDF)": engine_lex,
        "semantic (BGE)": engine,
        "hybrid (RRF)": hybrid,
    },
    qrels,
    k=10,
)

retriever          Recall@10     NDCG@10         MRR
----------------------------------------------------
lexical (TF-IDF)       0.350       0.227       0.189
semantic (BGE)         0.450       0.340       0.306
hybrid (RRF)           0.467       0.310       0.261


{'lexical (TF-IDF)': {'Recall@10': 0.35,
  'NDCG@10': 0.2266918544970872,
  'MRR': 0.18914021164021164},
 'semantic (BGE)': {'Recall@10': 0.45,
  'NDCG@10': 0.34027296625920733,
  'MRR': 0.3056812169312169},
 'hybrid (RRF)': {'Recall@10': 0.4666666666666667,
  'NDCG@10': 0.3095780104436432,
  'MRR': 0.26080853174603175}}

In [11]:
from collections import defaultdict


class WeightedHybrid:
    """
    Weighted Reciprocal Rank Fusion (RRF).

    Extends standard RRF by assigning different importance
    weights to individual retrievers.

    This allows ranking contributions from semantic and
    lexical retrieval systems to be tuned independently.
    """

    def __init__(self, retrievers_weights, rrf_k=60, pool=50):
        """
        Args:
            retrievers_weights:
                List of (retriever, weight) pairs.

            rrf_k:
                Rank smoothing parameter.

            pool:
                Candidate pool size from each retriever.
        """
        self.rw = retrievers_weights
        self.rrf_k = rrf_k
        self.pool = pool

    def search(self, query, k=10):
        """
        Fuse rankings from multiple retrievers using
        weighted reciprocal rank fusion.
        """

        scores = defaultdict(float)
        meta = {}

        for retriever, weight in self.rw:

            results = retriever.search(
                query,
                k=self.pool
            )

            for rank, hit in enumerate(results):

                # Weighted RRF contribution.
                scores[hit["id"]] += (
                    weight /
                    (self.rrf_k + rank + 1)
                )

                meta[hit["id"]] = hit

        ranked_ids = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                "id": doc_id,
                "score": scores[doc_id],
                **{
                    field: meta[doc_id][field]
                    for field in ("text", "metadata")
                    if field in meta[doc_id]
                }
            }
            for doc_id in ranked_ids
        ]


# -------------------------------------------------------
# Compare different fusion strategies.
#
# Goal:
#     Measure the impact of semantic-heavy versus
#     balanced retrieval fusion.
# -------------------------------------------------------
from evaluate import compare

compare(
    {
        "semantic (BGE)": engine,

        "hybrid 1:1":
            WeightedHybrid([
                (engine, 1),
                (engine_lex, 1)
            ]),

        "hybrid 2:1":
            WeightedHybrid([
                (engine, 2),
                (engine_lex, 1)
            ]),

        "hybrid 3:1":
            WeightedHybrid([
                (engine, 3),
                (engine_lex, 1)
            ]),
    },
    qrels,
    k=10
)

retriever          Recall@10     NDCG@10         MRR
----------------------------------------------------
semantic (BGE)         0.450       0.340       0.306
hybrid 1:1             0.467       0.310       0.261
hybrid 2:1             0.467       0.322       0.277
hybrid 3:1             0.458       0.327       0.285


{'semantic (BGE)': {'Recall@10': 0.45,
  'NDCG@10': 0.34027296625920733,
  'MRR': 0.3056812169312169},
 'hybrid 1:1': {'Recall@10': 0.4666666666666667,
  'NDCG@10': 0.3095780104436432,
  'MRR': 0.26080853174603175},
 'hybrid 2:1': {'Recall@10': 0.4666666666666667,
  'NDCG@10': 0.3223914368142084,
  'MRR': 0.27727017195767195},
 'hybrid 3:1': {'Recall@10': 0.4583333333333333,
  'NDCG@10': 0.32690417082985235,
  'MRR': 0.2854001322751323}}

In [12]:
from sentence_transformers import CrossEncoder

# Cross-encoder reranking model.
# Downloads pretrained weights on first use (~hundreds of MB).
reranker = CrossEncoder("BAAI/bge-reranker-base", device="cuda")


class RerankRetriever:
    """
    Two-stage retrieval pipeline:

    1. Retrieve a candidate pool using a fast retriever.
    2. Re-rank candidates with a cross-encoder.
    3. Return the top-k highest-scoring results.
    """

    def __init__(self, base, reranker, pool=50):
        self.base = base
        self.reranker = reranker
        self.pool = pool

    def search(self, query, k=10):

        # Stage 1: retrieve a larger candidate set.
        cands = self.base.search(query, k=self.pool)

        if not cands:
            return []

        # Build (query, document) pairs for cross-encoder scoring.
        pairs = [(query, c["text"]) for c in cands]

        # Stage 2: compute fine-grained relevance scores.
        scores = self.reranker.predict(pairs)

        # Sort candidates by reranker score.
        order = sorted(
            range(len(cands)),
            key=lambda i: scores[i],
            reverse=True
        )

        # Return top-k reranked results.
        out = []
        for i in order[:k]:
            c = dict(cands[i])
            c["score"] = float(scores[i])
            out.append(c)

        return out


# Hybrid retrieval:
# 75% semantic retrieval + 25% lexical retrieval.
hybrid_3to1 = WeightedHybrid([
    (engine, 3),
    (engine_lex, 1)
])

# Apply cross-encoder reranking on top of hybrid retrieval.
rerank = RerankRetriever(
    hybrid_3to1,
    reranker,
    pool=50
)

from evaluate import compare

compare({
    "semantic (BGE)": engine,
    "hybrid 3:1": hybrid_3to1,
    "hybrid 3:1 + rerank": rerank,
}, qrels, k=10)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

retriever          Recall@10     NDCG@10         MRR
----------------------------------------------------
semantic (BGE)         0.450       0.340       0.306
hybrid 3:1             0.458       0.327       0.285
hybrid 3:1 + rerank       0.537       0.414       0.375


{'semantic (BGE)': {'Recall@10': 0.45,
  'NDCG@10': 0.34027296625920733,
  'MRR': 0.3056812169312169},
 'hybrid 3:1': {'Recall@10': 0.4583333333333333,
  'NDCG@10': 0.32690417082985235,
  'MRR': 0.2854001322751323},
 'hybrid 3:1 + rerank': {'Recall@10': 0.5375,
  'NDCG@10': 0.41377739903489724,
  'MRR': 0.3747371031746031}}